In [ ]:
# ============================
# Imports & basic settings
# ============================

# Numeric + I/O
import os                          # filesystem operations (folders/files)
import json                        # save metadata in readable format
import numpy as np                 # numerical arrays

# MATLAB file loading
from scipy.io import loadmat       # to read .mat files

# Signal processing utilities
from scipy.signal import resample, butter, filtfilt

# Misc
import warnings
warnings.filterwarnings("ignore")  # keep the notebook output tidy



# ============================
# Pipeline configuration
# ============================

# ----------------------------
# Data folders
# ----------------------------

DATA_DIR = "data"
# location of original EEG .mat files (one subject per file)

OUT_DIR = "processed_user"
# window-level processed data will be saved here

os.makedirs(OUT_DIR, exist_ok=True)
# create output directory if it doesn't exist


# ----------------------------
# Filtering parameters
# ----------------------------

L_FREQ = 0.5
# lower cutoff at 0.5 Hz
# removes DC drift while keeping slow cortical activity

H_FREQ = 40.0
# upper cutoff at 40 Hz
# keeps major EEG bands (delta–beta)
# removes high-frequency muscle noise


# ----------------------------
# Windowing configuration
# ----------------------------

WINDOW_SEC = 2.0
# 2-second windows
# provides enough temporal context for identity patterns
# keeps sample count reasonably high

WINDOW_OVERLAP = 1.0
# 50% overlap
# increases training samples while preserving temporal continuity


# ----------------------------
# Resampling + Patch settings
# ----------------------------

TARGET_SF = 128
# resample EEG to 128 Hz
# sufficient for 0.5–40 Hz band
# reduces computational load

PATCH_LEN = 32
# each patch = 32 samples
# at 128 Hz → 0.25 sec per token
# balances temporal resolution and token count


# extract numeric EEG, sampling rate, and channel names
def load_eeg_from_mat(mat_path):
    """
    Load one .mat EEG file and extract:
    - signals → shape (n_samples, n_channels)
    - sfreq → sampling frequency (Hz)
    - ch_names → channel labels (fallback to generic names if missing)
    """

    mat = loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    # load MATLAB file with simplified nested structure
    # (why: makes field access easier and avoids extra nesting)

    if 'EEG' not in mat:
        raise ValueError("No 'EEG' key in mat file")
        # enforce expected structure
        # (why: prevents silent failure if file format is different)

    eeg = mat['EEG']

    signals = None
    timestamps = None
    ch_names = None


    # -------------------------
    # Robust field extraction
    # -------------------------

    if hasattr(eeg, 'dtype') and eeg.dtype.names:
        # structured numpy-style MATLAB object

        for nm in eeg.dtype.names:
            val = eeg[nm]
            # iterate through fields to locate numeric EEG matrix

            if isinstance(val, np.ndarray) and val.ndim == 2 and val.dtype.kind in 'fiu':
                signals = np.asarray(val)
                break
                # first valid 2D numeric matrix assumed as EEG signal
                # (why: EEG data stored as 2D samples × channels)

        if 'timestamp' in eeg.dtype.names:
            timestamps = np.asarray(eeg['timestamp']).flatten()
            # flatten timestamp vector
            # (why: easier interval computation for sfreq)

        if 'chanlocs' in eeg.dtype.names:
            try:
                ch_names = []
                for ch in np.atleast_1d(eeg['chanlocs']):
                    if hasattr(ch, 'labels'):
                        ch_names.append(str(ch.labels))
                # extract channel labels if available
                # (why: useful for interpretability and debugging)
            except Exception:
                ch_names = None
                # fallback if structure inconsistent
                # (why: avoid crashing on malformed metadata)

    else:
        # object-style MATLAB structure

        signals = getattr(eeg, 'data', None)
        # directly access data field
        # (why: some MATLAB exports store signals under .data)

        timestamps = getattr(eeg, 'timestamp', None)
        # attempt timestamp extraction

        chanlocs = getattr(eeg, 'chanlocs', None)

        if chanlocs is not None:
            try:
                ch_names = [str(c.labels) for c in chanlocs]
                # extract channel labels from object-style structure
            except Exception:
                ch_names = None
                # fallback if labels missing


    if signals is None:
        raise ValueError("Could not find numeric EEG data")
        # ensure signal matrix exists
        # (why: pipeline cannot proceed without numeric EEG)


    signals = np.asarray(signals, dtype=np.float64)
    # enforce consistent numeric precision
    # (why: prevents dtype inconsistencies during filtering and processing)


    if signals.ndim != 2:
        raise ValueError("EEG signals must be 2D (samples × channels or channels × samples)")
        # enforce strict 2D structure
        # (why: model expects time × channel matrix)


    # -------------------------
    # Orientation check
    # -------------------------

    if signals.shape[0] < signals.shape[1]:
        signals = signals.T
        # transpose to (samples × channels)
        # (why: typical EEG has many time samples and few channels; reversed shape breaks windowing)


    # -------------------------
    # Sampling frequency estimation
    # -------------------------

    sfreq = None

    if timestamps is not None and len(timestamps) >= 2:
        ts = np.asarray(timestamps, dtype=float)
        duration = ts[-1] - ts[0]

        if duration > 1e6:
            ts /= 1e6
            # convert microseconds to seconds
            # (why: sfreq must be computed in seconds)

        elif duration > 1e3:
            ts /= 1e3
            # convert milliseconds to seconds
            # (why: ensure correct time unit before frequency calculation)

        diffs = np.diff(ts)
        diffs = diffs[diffs > 0]
        # remove non-positive intervals
        # (why: avoid invalid or duplicate timestamps)

        if len(diffs):
            sfreq = 1.0 / np.mean(diffs)
            # compute sampling rate from mean interval
            # (why: averaging reduces timestamp jitter noise)


    # fallback: check common metadata fields
    for key in ('fs', 'srate', 'sfreq', 'sampling_rate'):
        if sfreq is None and key in mat:
            try:
                sfreq = float(mat[key])
                # use metadata sampling rate if available
                # (why: more reliable than timestamp estimation)
            except Exception:
                pass


    if sfreq is None or not (1.0 < sfreq < 2000.0):
        sfreq = 220.0
        # fallback default sampling rate
        # (why: prevents pipeline crash if metadata missing or corrupted)


    # -------------------------
    # Channel name fallback
    # -------------------------

    if ch_names is None or len(ch_names) != signals.shape[1]:
        ch_names = [f"EEG{c+1}" for c in range(signals.shape[1])]
        # generate generic channel names
        # (why: downstream steps expect channel labels even if file lacks metadata)


    return signals, sfreq, ch_names


# convert µV → V heuristic
def ensure_volts_unit(signals):
    """
    Ensure EEG signals are in volts.
    If median magnitude suggests microvolts, convert to volts.
    """

    signals = np.asarray(signals, dtype=np.float64)
    # enforce consistent numeric precision
    # (why: avoids dtype issues during filtering and scaling)


    med = np.nanmedian(np.abs(signals))
    # compute median absolute amplitude (NaN-safe)
    # (why: median is robust to outliers; absolute value reflects signal magnitude)


    if med > 1e-3:
        return signals * 1e-6
        # convert microvolts → volts
        # (why: typical EEG in µV has values around 10–100; in volts this becomes 1e-5 range)
        # (why: filtering functions expect signals in volt scale)

    return signals
    # return unchanged if already in volts
    # (why: prevents double scaling if data already correct)

# scipy.resample wrapper

def resample_if_needed(signals, orig_sfreq, target_sfreq=TARGET_SF):
    """
    Resample signals (n_samples, n_channels) 
    from orig_sfreq -> target_sfreq.
    Returns (resampled_signals, new_sfreq).
    """

    signals = np.asarray(signals, dtype=np.float64)
    # ensure consistent float precision
    # (why: FFT-based resampling expects numeric stability and uniform dtype)


    if abs(orig_sfreq - target_sfreq) < 1e-6:
        return signals, float(orig_sfreq)
        # skip resampling if already at target frequency
        # (why: avoids unnecessary interpolation and preserves original signal)


    num_samples = int(round(signals.shape[0] * target_sfreq / orig_sfreq))
    # compute new number of time samples after resampling
    # (why: time axis must scale proportionally to frequency ratio)


    if num_samples < 2:
        raise ValueError("Resulting number of samples < 2 after resampling")
        # safety check to prevent invalid signal length
        # (why: downstream windowing requires meaningful time dimension)


    resig = resample(signals, num_samples, axis=0)
    # perform FFT-based resampling along time axis
    # (why: scipy.resample uses Fourier interpolation for efficient frequency-domain scaling)


    if np.iscomplexobj(resig):
        resig = np.real(resig)
        # remove small imaginary components introduced by FFT
        # (why: numerical precision may create negligible complex values)


    return resig.astype(np.float64), float(target_sfreq)
    # return resampled signal and updated sampling rate
    # (why: keep dtype consistent and ensure sfreq matches new temporal resolution)

# Cell 3d: notch + bandpass

def apply_bandpass_filter(signals, sfreq, l_freq=L_FREQ, h_freq=H_FREQ):
    """
    Apply bandpass filter to signals using SciPy.
    signals shape: (n_samples, n_channels)
    """

    nyq = 0.5 * sfreq
    # compute Nyquist frequency
    # (why: digital filter cutoff frequencies must be normalized relative to Nyquist)


    low = l_freq / nyq
    high = h_freq / nyq
    # normalize cutoff frequencies
    # (why: scipy butter expects values between 0 and 1, relative to Nyquist)


    # 4th-order Butterworth is a good compromise
    b, a = butter(4, [low, high], btype='band')
    # design 4th-order bandpass filter
    # (why: Butterworth gives smooth frequency response without ripples)
    # (why: order 4 balances sharpness and stability)


    return filtfilt(b, a, signals, axis=0)
    # apply zero-phase filtering
    # (why: filtfilt prevents phase distortion by filtering forward and backward)
    # (why: preserve temporal structure for identity features)



def apply_notch_filter(signals, sfreq, notch_freq=50.0, q=30.0):
    """
    Apply notch filter at a single frequency.
    """

    nyq = 0.5 * sfreq
    # compute Nyquist frequency
    # (why: needed for frequency normalization)


    w0 = notch_freq / nyq
    # normalize notch center frequency
    # (why: butter requires normalized frequency input)


    b, a = butter(2, [w0 - w0/q, w0 + w0/q], btype='bandstop')
    # design narrow bandstop filter around target frequency
    # (why: removes powerline interference at 50 Hz)
    # (why: q controls notch width; higher q = narrower notch)


    return filtfilt(b, a, signals, axis=0)
    # apply zero-phase filtering
    # (why: avoids phase shift in EEG signal)

# Patch Extraction for Transformer Input
def patchify(window, patch_len=32):
    """
    Split a window into non-overlapping patches.

    window: (channels, time)
    returns: (num_patches, channels * patch_len)

    Each patch is flattened so it can be treated as one token
    for the Transformer encoder.
    """

    # window shape is (C, T)
    # C = number of EEG channels
    # T = number of time samples in this window
    C, T = window.shape

    patches = []  # store each patch as a flattened token

    # iterate over time axis in steps of patch_len
    # (why: non-overlapping patches → cleaner temporal segmentation)
    for start in range(0, T - patch_len + 1, patch_len):

        # slice a temporal segment of length patch_len
        # shape becomes (channels, patch_len)
        patch = window[:, start:start + patch_len]

        # flatten into 1D vector
        # (why: Transformer expects tokens as vectors (not 2D matrices))
        # final size = channels × patch_len
        patches.append(patch.reshape(-1))

    # stack into final tensor
    # shape = (num_patches, channels * patch_len)
    return np.stack(patches)

# Identity-preserving preprocessing + window patch extraction

from tqdm import tqdm
# progress bar for tracking preprocessing status
# (why: makes long preprocessing transparent and easier to debug)

WINDOW_SAMPLES = int(WINDOW_SEC * TARGET_SF)
# number of time samples per window
# (why: 2 sec × 128 Hz = 256 samples per identity window)

STEP_SAMPLES = int((WINDOW_SEC - WINDOW_OVERLAP) * TARGET_SF)
# stride between consecutive windows
# (why: controls overlap; 50% overlap improves data density without full redundancy)

os.makedirs(OUT_DIR, exist_ok=True)
# ensure output directory exists
# (why: prevents crash during saving if folder missing)


mat_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.mat')])
# collect all subject EEG files
# (why: each file corresponds to one user identity)

print(f"Found {len(mat_files)} MAT files in {DATA_DIR}")


all_tokens = []
# stores patch tokens for all windows of all users
# (why: this becomes model input tensor X)

all_labels = []
# stores corresponding user IDs
# (why: this becomes label vector y)

all_files = []
# optional tracking of file origin
# (why: helps debug subject-level issues)


for fname in tqdm(mat_files, disable=True):
    #desc="Processing EEG files", unit="file"):

    mat_path = os.path.join(DATA_DIR, fname)
    # construct full file path


    try:
        # 1) Load EEG
        signals, sfreq, ch_names = load_eeg_from_mat(mat_path)
        # extract numeric signal matrix
        # (why: converts raw MATLAB structure to usable NumPy array)

        # 2) Unit correction
        signals = ensure_volts_unit(signals)
        # convert µV → V if necessary
        # (why: keeps amplitude scale consistent across subjects)

        # 3) Resample
        signals, sfreq = resample_if_needed(signals, sfreq, TARGET_SF)
        # unify sampling rate to 128 Hz
        # (why: Transformer expects consistent temporal resolution)

        # 4) Bandpass filter
        signals = apply_bandpass_filter(signals, sfreq)
        # keep 0.5–40 Hz brain rhythms
        # (why: removes DC drift and high-frequency noise while preserving identity-relevant bands)


        n_samples, n_channels = signals.shape
        # retrieve signal dimensions
        # (why: used for sliding window iteration)


        # 5) Sliding window + patch extraction
        for start in range(0, n_samples - WINDOW_SAMPLES + 1, STEP_SAMPLES):

            window = signals[start:start + WINDOW_SAMPLES].T
            # extract time segment and transpose to (channels, time)
            # (why: patchify expects channel-first format)


            window = (window - window.mean(axis=1, keepdims=True)) / (
                window.std(axis=1, keepdims=True) + 1e-6
            )
            # per-channel z-score normalization
            # (why: removes amplitude bias while preserving temporal structure)


            patches = patchify(window, patch_len=32)
            # split window into fixed-length patches
            # (why: Transformer processes local temporal chunks instead of raw full sequence)


            all_tokens.append(patches)
            # store patch tokens for this window
            # (why: each window becomes one training sample)


            user_id = int(fname.replace("Subject", "").replace(".mat", ""))
            # extract numeric subject ID from filename
            # (why: supervised identity classification requires label mapping)


            all_labels.append(user_id)
            # store corresponding user label

            all_files.append(fname)
            # optional bookkeeping


    except Exception as e:
        print(f"[ERROR] {fname}: {e}")
        continue
        # skip corrupted files but continue processing
        # (why: prevents one bad file from stopping entire pipeline)



# Convert to arrays
X = np.array(all_tokens, dtype=np.float32)
# convert list → 3D tensor
# (why: model training requires fixed numeric tensor format)

y = np.array(all_labels, dtype=np.int64)
# convert labels to integer vector
# (why: CrossEntropyLoss expects int64 class labels)


print("Final token tensor shape:", X.shape)
print("Unique users:", len(np.unique(y)))
# sanity checks
# (why: confirms preprocessing pipeline correctness before training)


np.save(os.path.join(OUT_DIR, "X_tokens.npy"), X)
# save processed features
# (why: avoids recomputation during future training runs)

np.save(os.path.join(OUT_DIR, "y_labels.npy"), y)
# save corresponding labels


print(np.load("processed_user/X_tokens.npy").shape)
# quick reload check
# (why: ensures file was written correctly)


print("Identity tokens saved. Ready for Transformer training.")

In [ ]:
# Pre-training Visual Diagnostics
import matplotlib.pyplot as plt
from scipy.signal import welch
from sklearn.decomposition import PCA

plt.rcParams.update({
    "figure.figsize": (10, 4),
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

# Plot raw vs filtered signal
def plot_raw_vs_filtered(signals_raw, signals_filtered, sfreq, channel=0):
    """
    Compare raw and filtered EEG signals for one channel
    using vertically stacked subplots (cleaner visual comparison).
    """

    segment_duration = 10  # seconds to visualize
    # why: plotting entire recording (~1000s) hides waveform details

    segment_samples = int(segment_duration * sfreq)
    # convert seconds → number of samples

    t = np.arange(segment_samples) / sfreq
    # time axis for selected segment

    fig, axes = plt.subplots(2, 1, sharex=True)
    # 2 rows, 1 column
    # sharex=True ensures both plots use same time axis

    # --- Raw Signal ---
    axes[0].plot(t, signals_raw[:segment_samples, channel], alpha=0.8)
    axes[0].set_title(f"Raw Signal (Channel {channel+1})")
    axes[0].set_ylabel("Amplitude (V)")
    # raw contains DC offset + drift + noise

    # --- Filtered Signal ---
    axes[1].plot(t, signals_filtered[:segment_samples, channel], alpha=0.8)
    axes[1].set_title("Bandpass Filtered Signal (0.5–40 Hz)")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Amplitude (V)")
    # filtered removes DC drift + high-frequency noise
    # signal should now be centered around zero

    plt.tight_layout()
    plt.savefig("raw_vs_bandpass.png")
    plt.show()

sample_file = mat_files[0]
# select first subject file for visualization

signals_raw, sfreq, _ = load_eeg_from_mat(os.path.join(DATA_DIR, sample_file))
# load numeric EEG

signals_raw = ensure_volts_unit(signals_raw)
# convert µV → V if necessary

signals_resampled, sfreq = resample_if_needed(signals_raw, sfreq, TARGET_SF)
# enforce uniform sampling rate (128 Hz)

signals_filtered = apply_bandpass_filter(signals_resampled, sfreq)
# remove frequencies outside 0.5–40 Hz

plot_raw_vs_filtered(signals_resampled, signals_filtered, sfreq)
# compare before vs after filtering

# Plot PSD comparison
def plot_psd_comparison(raw_signal, filtered_signal, sfreq, channel=0):
    """
    Compare power spectral density before and after filtering.
    (why: verify that frequencies outside 0.5–40 Hz are suppressed)
    """

    f_raw, Pxx_raw = welch(raw_signal[:, channel], sfreq)
    # compute PSD of raw signal using Welch method
    # (why: frequency-domain view of energy distribution)

    f_filt, Pxx_filt = welch(filtered_signal[:, channel], sfreq)
    # compute PSD of filtered signal
    # (why: confirm unwanted frequency components reduced)

    plt.figure()
    # create new figure

    plt.semilogy(f_raw, Pxx_raw, label="Raw PSD")
    # log-scale y-axis for better dynamic range visibility

    plt.semilogy(f_filt, Pxx_filt, label="Filtered PSD")
    # overlay filtered PSD for comparison

    plt.title("Power Spectral Density Comparison")
    # descriptive title

    plt.xlabel("Frequency (Hz)")
    # x-axis is frequency in Hz

    plt.ylabel("Power")
    # power spectral density magnitude

    plt.legend()
    # show legend

    plt.tight_layout()
    # prevent label clipping

    plt.savefig("psd.png")
    plt.show()
    # display figure

plot_psd_comparison(signals_resampled, signals_filtered, sfreq)

# Plot sliding window segmentation
def plot_window_boundaries(signal, sfreq, window_samples, step_samples):
    """
    Visualize sliding window segmentation.
    (why: confirm overlap and step size are correct)
    """

    t = np.arange(signal.shape[0]) / sfreq
    # convert sample indices to time axis (seconds)

    plt.figure()
    # create figure

    plt.plot(t, signal[:, 0], alpha=0.7)
    # plot first channel only
    # (why: single channel sufficient to visualize segmentation)

    for start in range(0, signal.shape[0] - window_samples + 1, step_samples):
        plt.axvline(start / sfreq, linestyle='--', alpha=0.4)
        # draw vertical line at each window start
        # (why: visually show window boundaries and overlap)

    plt.title("Sliding Window Segmentation (Channel 1)")
    # descriptive title

    plt.xlabel("Time (s)")
    # time axis

    plt.ylabel("Amplitude")
    # signal amplitude

    plt.tight_layout()
    # adjust layout

    plt.savefig("sliding_window.png")
    plt.show()
    # display plot
    
plot_window_boundaries(signals_filtered, sfreq, WINDOW_SAMPLES, STEP_SAMPLES)

# Plot windows per subject
def plot_windows_per_subject(labels):
    """
    Show how many windows each subject contributes.
    (why: ensure dataset is balanced across users)
    """

    unique, counts = np.unique(labels, return_counts=True)
    # count number of windows per subject

    plt.figure()
    # create figure

    plt.bar(unique, counts)
    # bar plot of subject vs window count

    plt.title("Number of Windows per Subject")
    # descriptive title

    plt.xlabel("Subject ID")
    # subject index

    plt.ylabel("Window Count")
    # number of windows

    plt.tight_layout()
    # adjust layout
    
    plt.savefig("win_per_sub.png")
    plt.show()
    # display plot

plot_windows_per_subject(y)

# Plot heatmap
def plot_patch_heatmap(window, patch_len=32):
    """
    Visualize one patch as a heatmap (channels × time).
    (why: inspect structure of token input to Transformer)
    """

    patches = patchify(window, patch_len)
    # split window into temporal patches

    C = window.shape[0]
    # number of channels

    first_patch = patches[0].reshape(C, patch_len)
    # reshape first flattened patch back to (channels × patch_len)
    # (why: easier to interpret spatial-temporal structure)

    plt.figure()
    # create figure

    plt.imshow(first_patch, aspect='auto')
    # visualize amplitude values as color intensity

    plt.title("First Patch Heatmap (Channels × Time)")
    # descriptive title

    plt.xlabel("Time Samples")
    # patch time axis

    plt.ylabel("Channels")
    # channel axis

    plt.colorbar()
    # show amplitude scale

    plt.tight_layout()
    # adjust layout
    
    plt.savefig("first_patch_heatmap.png")
    plt.show()
    # display plot

window = signals_filtered[:WINDOW_SAMPLES].T
window = (window - window.mean(axis=1, keepdims=True)) / (
    window.std(axis=1, keepdims=True) + 1e-6
)

plot_patch_heatmap(window, PATCH_LEN)

# Plot patch boundaries

def plot_window_with_patches(window, sfreq, patch_len):
    """
    Visualize one EEG window and show patch boundaries.
    window shape: (channels, time_samples)
    (why: confirm correct token segmentation before training)
    """

    n_samples = window.shape[1]
    # total time samples in window

    time_axis = np.arange(n_samples) / sfreq
    # convert sample index to seconds

    plt.figure(figsize=(10, 4))
    # create readable figure size

    plt.plot(time_axis, window[0], label="Channel 1")
    # plot first channel only
    # (why: segmentation visibility sufficient with one channel)

    for p in range(0, n_samples, patch_len):
        plt.axvline(p / sfreq, linestyle='--', alpha=0.5)
        # draw vertical boundary for each patch
        # (why: show how 2-sec window splits into tokens)

    plt.xlabel("Time (s)")
    # x-axis label

    plt.ylabel("Amplitude")
    # y-axis label

    plt.title("EEG Window with Patch Boundaries")
    # descriptive title

    plt.legend()
    # show legend

    plt.tight_layout()
    # prevent clipping
    
    plt.savefig("patch_boundaries.png")
    plt.show()
    # display figure

sample_file = mat_files[0]
# select first subject file for demonstration
# (why: use a fixed example file for consistent visualization)


signals, sfreq, _ = load_eeg_from_mat(os.path.join(DATA_DIR, sample_file))
# load raw EEG signal and sampling frequency
# ignore channel names using "_"
# (why: channel labels not required for this visualization)


signals, sfreq = resample_if_needed(signals, sfreq, TARGET_SF)
# resample signal to target frequency (128 Hz)
# (why: ensure window size matches training configuration)


window = signals[:256].T
# extract first 256 samples (2 seconds at 128 Hz)
# transpose to (channels × time)
# (why: visualization function expects channels first)


plot_window_with_patches(window, sfreq, PATCH_LEN)
# plot signal with vertical patch boundaries
# (why: visually confirm 2-sec window is split into correct patch segments)

# Plot PCA
from sklearn.decomposition import PCA

def plot_pca_projection(X, y):
    """
    Project high-dimensional patch features into 2D using PCA.
    Visualizes how well subjects separate before model training.
    """

    # Flatten each sample to a single feature vector
    X_flat = X.reshape(X.shape[0], -1)
    # (why: PCA expects 2D input → [samples × features])

    pca = PCA(n_components=2)
    # reduce dimensionality to 2 components
    # (why: 2D projection allows visual inspection of feature clustering)

    X_2d = pca.fit_transform(X_flat)
    # compute principal components and project data
    # (why: capture maximum variance directions in feature space)

    plt.figure()
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, s=5)
    # color by subject label
    # (why: visually inspect inter-subject separability)

    plt.title("PCA Projection of Pre-Training Features")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.tight_layout()
    plt.savefig("pca_projection.png")
    plt.show()

plot_pca_projection(X, y)

# Plot subject centroid distance

from scipy.spatial.distance import cdist

def plot_subject_centroid_distance(X, y):
    """
    Compute distance between subject feature centroids.
    Shows how distinct subjects are in feature space.
    """

    X_flat = X.reshape(X.shape[0], -1)
    # (why: flatten patches into feature vectors)

    subjects = np.unique(y)
    # list of subject IDs
    # (why: compute one centroid per subject)

    centroids = []

    for subj in subjects:
        subj_features = X_flat[y == subj]
        # extract features belonging to one subject
        # (why: compute subject-specific mean representation)

        centroid = subj_features.mean(axis=0)
        # average feature vector
        # (why: represents subject signature in feature space)

        centroids.append(centroid)

    centroids = np.stack(centroids)

    distance_matrix = cdist(centroids, centroids)
    # pairwise Euclidean distance between subjects
    # (why: larger distance → better inter-subject separability)

    plt.figure()
    plt.imshow(distance_matrix, aspect='auto')
    plt.colorbar()
    plt.title("Inter-Subject Centroid Distance Matrix")
    plt.xlabel("Subject Index")
    plt.ylabel("Subject Index")
    plt.tight_layout()
    plt.savefig("centroid.png")
    plt.show()

plot_subject_centroid_distance(X, y)

In [ ]:
# Setup and Configuration
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import math
import json
import random
import time
from collections import defaultdict
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold
from sklearn.manifold import TSNE
from scipy.spatial.distance import cdist
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
import matplotlib.pyplot as plt
import wandb

# Use all GPUs
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"Available GPUs: {torch.cuda.device_count()}")

# Configuration
SEED = 42
BATCH_SIZE = 32  # Larger batch for DataParallel
EPOCHS_PER_FOLD = 150
BASE_LR = 1e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 5
D_MODEL = 192
N_HEADS = 6
NUM_LAYERS = 3
DROPOUT = 0.3
PATCH_LEN = 32
DATA_DIR = "processed_user"
TOKEN_DIM = PATCH_LEN * 4
WINDOW_SAMPLES = 256
NUM_WORKERS = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
# Model Definitions
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class EEGTokenTransformer(nn.Module):
    def __init__(self, token_dim=16, num_tokens=8, d_model=192, n_heads=6,
                 num_layers=3, num_classes=171, dropout=0.3):
        super().__init__()
        self.token_embed = nn.Linear(token_dim, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        self.channel_weights = nn.Parameter(torch.ones(token_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.attn_pool = nn.Linear(d_model, 1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )
    def forward(self, x):
        x = x * self.channel_weights
        x = self.token_embed(x)
        x = self.pos_encoding(x)
        x = self.encoder(x)
        weights = torch.softmax(self.attn_pool(x), dim=1)
        x = torch.sum(weights * x, dim=1)
        x = self.norm(x)
        x = F.normalize(x, dim=1)
        logits = self.classifier(x)
        return logits, x

class ArcFaceLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, s=30.0, m=0.25):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.randn(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, embeddings, labels):
        labels = labels.to(embeddings.device)
        embeddings = F.normalize(embeddings, dim=1)
        weights = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings, weights)
        theta = torch.acos(torch.clamp(cosine, -1 + 1e-7, 1 - 1e-7))
        target_cos = torch.cos(theta + self.m)
        one_hot = F.one_hot(labels, num_classes=cosine.size(1)).float()
        logits = cosine * (1 - one_hot) + target_cos * one_hot
        logits *= self.s
        return F.cross_entropy(logits, labels)

def get_warmup_cosine_scheduler(optimizer, warmup_epochs, max_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))
        progress = float(epoch - warmup_epochs) / float(max(1, max_epochs - warmup_epochs))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
# Dataset and Helper Functions
class EEGTokenDataset(Dataset):
    def __init__(self, data_dir):
        self.X = np.load(os.path.join(data_dir, "X_tokens.npy"))
        y_raw = np.load(os.path.join(data_dir, "y_labels.npy"))
        unique_users = np.unique(y_raw)
        self.user_to_class = {u: i for i, u in enumerate(unique_users)}
        self.y = np.array([self.user_to_class[u] for u in y_raw], dtype=np.int64)
        print(f"Loaded {len(self.X)} samples, {len(unique_users)} classes")
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

def compute_strict_crr(all_preds, all_labels, threshold=0.6):
    subject_correct = defaultdict(list)
    for pred, true in zip(all_preds, all_labels):
        subject_correct[true].append(1 if pred == true else 0)
    correct = sum(1 for lst in subject_correct.values() if np.mean(lst) >= threshold)
    return correct / len(subject_correct) if subject_correct else 0

def build_templates(embeddings, labels):
    templates = {}
    for label in np.unique(labels):
        mask = labels == label
        if mask.sum() >= 1:
            templates[label] = embeddings[mask].mean(axis=0)
            templates[label] /= (np.linalg.norm(templates[label]) + 1e-8)
    return templates

def compute_eer_from_scores(genuine, impostor):
    thresholds = np.linspace(-1, 1, 500)
    best_eer, best_thresh = 1.0, 0.0
    for t in thresholds:
        far = np.mean(impostor >= t)
        frr = np.mean(genuine < t)
        eer = (far + frr) / 2
        if eer < best_eer:
            best_eer, best_thresh = eer, t
    return best_eer, best_thresh

def compute_eer_proper(model, train_loader, val_loader, device):
    """Compute EER using templates from TRAINING set, test on VALIDATION set"""
    model.eval()
    
    # Extract training embeddings for templates
    train_embs, train_lbls = [], []
    with torch.no_grad():
        for X, y in train_loader:
            X = X.to(device)
            _, emb = model(X)
            train_embs.append(emb.cpu().numpy())
            train_lbls.append(y.cpu().numpy())
    train_embs = np.concatenate(train_embs)
    train_lbls = np.concatenate(train_lbls)
    train_embs /= (np.linalg.norm(train_embs, axis=1, keepdims=True) + 1e-8)
    
    # Extract validation embeddings
    val_embs, val_lbls = [], []
    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            _, emb = model(X)
            val_embs.append(emb.cpu().numpy())
            val_lbls.append(y.cpu().numpy())
    val_embs = np.concatenate(val_embs)
    val_lbls = np.concatenate(val_lbls)
    val_embs /= (np.linalg.norm(val_embs, axis=1, keepdims=True) + 1e-8)
    
    # Build templates from TRAINING set
    templates = build_templates(train_embs, train_lbls)
    if len(templates) == 0:
        return None
    
    # Compute similarity on VALIDATION set
    template_matrix = np.stack(list(templates.values()))
    sim_matrix = cosine_similarity(val_embs, template_matrix)
    
    subj_to_idx = {s: i for i, s in enumerate(templates.keys())}
    genuine, impostor = [], []
    for i, label in enumerate(val_lbls):
        if label in subj_to_idx:
            true_idx = subj_to_idx[label]
            genuine.append(sim_matrix[i, true_idx])
            for j in range(len(templates)):
                if j != true_idx:
                    impostor.append(sim_matrix[i, j])
    
    if len(genuine) == 0 or len(impostor) == 0:
        return None
    
    eer, _ = compute_eer_from_scores(np.array(genuine), np.array(impostor))
    return eer

In [ ]:
# Load Dataset
dataset = EEGTokenDataset(DATA_DIR)
labels = dataset.y
num_classes = len(np.unique(labels))
print(f"Total samples: {len(dataset)}, Classes: {num_classes}")

# Create 5 folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(dataset.X, labels))

print(f"Created {len(folds)} folds")
for i, (train_idx, val_idx) in enumerate(folds):
    print(f"Fold {i+1}: Train={len(train_idx)}, Val={len(val_idx)}")

In [ ]:
# Train One Fold Function
from tqdm import tqdm
import time

def train_one_fold(fold_id, train_idx, val_idx):
    print(f"\n{'='*60}")
    print(f"TRAINING FOLD {fold_id+1}/5")
    print(f"{'='*60}")
    
    # Create subsets
    train_set = Subset(dataset, train_idx)
    val_set = Subset(dataset, val_idx)
    
    # Balanced sampler for training
    train_labels_fold = labels[train_idx]
    class_sample_count = np.bincount(train_labels_fold)
    weights = 1.0 / (class_sample_count + 1e-8)
    sample_weights = weights[train_labels_fold]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    
    # DataLoaders
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    
    # Model with DataParallel
    model = EEGTokenTransformer(
        token_dim=TOKEN_DIM, num_tokens=8, d_model=D_MODEL, n_heads=N_HEADS,
        num_layers=NUM_LAYERS, num_classes=num_classes, dropout=DROPOUT
    )
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
        model = nn.DataParallel(model)
    
    model = model.to(DEVICE)
    
    optimizer = optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_warmup_cosine_scheduler(optimizer, WARMUP_EPOCHS, EPOCHS_PER_FOLD)
    criterion = ArcFaceLoss(embedding_dim=D_MODEL, num_classes=num_classes).to(DEVICE)
    
    wandb.init(project="eeg-auth", name=f"kfold_fold_{fold_id+1}", reinit='finish_previous')
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_crr': [], 'val_eer': []}
    best_eer, best_eer_epoch = float('inf'), 0
    best_acc, best_crr = 0.0, 0.0
    
    for epoch in range(EPOCHS_PER_FOLD):
        epoch_start = time.time()
        
        # ==================== TRAINING ====================
        model.train()
        train_losses, train_preds, train_trues = [], [], []
        
        for X, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS_PER_FOLD} Train", leave=False):
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits, emb = model(X)
            loss = 0.7 * F.cross_entropy(logits, y, label_smoothing=0.05) + 0.3 * criterion(emb, y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            train_preds.extend(logits.argmax(dim=1).cpu().numpy())
            train_trues.extend(y.cpu().numpy())
        
        train_loss = np.mean(train_losses)
        train_acc = accuracy_score(train_trues, train_preds)
        
        # ==================== VALIDATION ====================  
        model.eval()
        val_losses, val_preds, val_trues = [], [], []

        with torch.no_grad():
            for X, y in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS_PER_FOLD} Val", leave=False):
                X, y = X.to(DEVICE), y.to(DEVICE)
                logits, emb = model(X)
                loss = 0.7 * F.cross_entropy(logits, y) + 0.3 * criterion(emb, y)
                val_losses.append(loss.item())
                val_preds.extend(logits.argmax(dim=1).cpu().numpy())
                val_trues.extend(y.cpu().numpy())
        
        val_loss = np.mean(val_losses)
        val_acc = accuracy_score(val_trues, val_preds)
        val_crr = compute_strict_crr(val_preds, val_trues, 0.6)
        
        # ==================== EER (every 5 epochs) ====================
        eer = None
        
        if epoch % 5 == 0 and epoch >= 10:
        
            eer = compute_eer_proper(
                model,
                train_loader,
                val_loader,
                DEVICE
            )
        
            if eer is not None and eer < best_eer:
        
                best_eer = eer
                best_eer_epoch = epoch + 1
        
                torch.save(
                    model.module.state_dict()
                    if hasattr(model, "module")
                    else model.state_dict(),
                    f"kfold_fold_{fold_id+1}_best_eer.pt"
                )
        
                print(
                    f"\n  Fold {fold_id+1} - New best EER: "
                    f"{eer*100:.4f}% at epoch {epoch+1}"
                )
        # Save best models
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.module.state_dict() if hasattr(model, 'module') else model.state_dict(),
                      f"kfold_fold_{fold_id+1}_best_acc.pt")
        if val_crr > best_crr:
            best_crr = val_crr
            torch.save(model.module.state_dict() if hasattr(model, 'module') else model.state_dict(),
                      f"kfold_fold_{fold_id+1}_best_crr.pt")
        
        # Store history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_crr'].append(val_crr)
        history['val_eer'].append(eer)
        
        wandb.log({
            'train_loss': train_loss,
            'val_loss': val_loss,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'val_crr': val_crr,
            'val_eer': eer if eer is not None else 0,
            'epoch': epoch + 1,
            'best_eer': best_eer,
        })
        
        scheduler.step()
        epoch_time = time.time() - epoch_start
        
        # Single line print per epoch (clean)
        print(f"  Epoch {epoch+1:2d}/{EPOCHS_PER_FOLD} | Loss: {train_loss:.4f} | Acc: {val_acc*100:.2f}% | CRR: {val_crr*100:.2f}% | Best EER: {best_eer*100:.2f}% | {epoch_time:.1f}s")
        
        if (epoch + 1) % 10 == 0:
            print(f"  {'-'*55}")
    
    with open(f"kfold_fold_{fold_id+1}_history.json", "w") as f:
        json.dump({k: [float(x) if x is not None else None for x in v] for k, v in history.items()}, f, indent=2)
    
    wandb.finish()
    
    print(f"\n{'='*50}")
    print(f"FOLD {fold_id+1} COMPLETED!")
    print(f"{'='*50}")
    print(f"   Best EER: {best_eer*100:.2f}% (epoch {best_eer_epoch})")
    print(f"   Best Acc: {best_acc*100:.2f}%")
    print(f"   Best CRR: {best_crr*100:.2f}%")
    print(f"{'='*50}\n")
    
    return {
    'fold': fold_id + 1,
    'eer': best_eer,
    'eer_epoch': best_eer_epoch,
    'val_acc': best_acc,
    'crr_60': best_crr
}

In [ ]:
# Run Fold 1
result_1 = train_one_fold(0, folds[0][0], folds[0][1])

In [ ]:
# Run Fold 2
result_2 = train_one_fold(1, folds[1][0], folds[1][1])

In [ ]:
# Run Fold 3
result_3 = train_one_fold(2, folds[2][0], folds[2][1])

In [ ]:
# Run Fold 4
result_4 = train_one_fold(3, folds[3][0], folds[3][1])

In [ ]:
# Run Fold 5
result_5 = train_one_fold(4, folds[4][0], folds[4][1])

In [ ]:
# Aggregate Results
results = [result_1, result_2, result_3, result_4, result_5]

print("\n" + "="*60)
print("K-FOLD SUMMARY")
print("="*60)
print(f"{'Fold':<6} {'EER (%)':<12} {'Val Acc (%)':<14} {'CRR@60% (%)':<14}")
print("-"*50)

eers, accs, crrs = [], [], []
for r in results:
    eers.append(r['eer'])
    accs.append(r['val_acc'])
    crrs.append(r['crr_60'])
    print(f"{r['fold']:<6} {r['eer']*100:<12.2f} {r['val_acc']*100:<14.2f} {r['crr_60']*100:<14.2f}")

print("-"*50)
print(f"{'Mean':<6} {np.mean(eers)*100:<12.2f} {np.mean(accs)*100:<14.2f} {np.mean(crrs)*100:<14.2f}")
print(f"{'Std':<6} {np.std(eers)*100:<12.2f} {np.std(accs)*100:<14.2f} {np.std(crrs)*100:<14.2f}")
print("="*60)

# Save master results
master_results = {
    'summary': {
        'eer_mean': float(np.mean(eers)), 'eer_std': float(np.std(eers)),
        'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs)),
        'crr_mean': float(np.mean(crrs)), 'crr_std': float(np.std(crrs))
    },
    'folds': results
}
with open("kfold_master_results.json", "w") as f:
    json.dump(master_results, f, indent=2)

In [ ]:
# Generate t-SNE for Best Fold 
# Find best fold
with open("kfold_master_results.json", "r") as f:
    master = json.load(f)
best_fold = min(master['folds'], key=lambda x: x['eer'])
best_fold_id = best_fold['fold']
print(f"Best fold: {best_fold_id} with EER = {best_fold['eer']*100:.2f}%")

# Load best model
model = EEGTokenTransformer(
    token_dim=TOKEN_DIM, num_tokens=8, d_model=D_MODEL, n_heads=N_HEADS,
    num_layers=NUM_LAYERS, num_classes=num_classes, dropout=DROPOUT
)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.load_state_dict(torch.load(f"kfold_fold_{best_fold_id}_best_eer.pt", map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# Get validation loader for best fold
fold_idx = best_fold_id - 1
val_set = Subset(dataset, folds[fold_idx][1])
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Extract embeddings
all_embs, all_labels = [], []
with torch.no_grad():
    for X, y in val_loader:
        X = X.to(DEVICE)
        _, emb = model(X)
        all_embs.append(emb.cpu().numpy())
        all_labels.append(y.numpy())
embeddings = np.concatenate(all_embs)
labels = np.concatenate(all_labels)
embeddings /= (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8)

# t-SNE
print("Running t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
embeddings_2d = tsne.fit_transform(embeddings[:3000])  # Limit to 3000 points

plt.figure(figsize=(12, 10))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=labels[:3000], s=10, cmap='tab20', alpha=0.7)
plt.colorbar(scatter, label='Subject ID')
plt.title(f'Fold {best_fold_id} - t-SNE of EEG Embeddings (EER={best_fold["eer"]*100:.2f}%)')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.savefig(f"kfold_best_fold_{best_fold_id}_tsne.png", dpi=150)
plt.show()